In [ ]:
% load_ext autoreload
% autoreload 2
import os

import numpy as np
import pandas as pd

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 50)
import sklearn as sk

import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams['figure.dpi'] = 250

# change working directory to project root
if os.getcwd().split('/')[-1] == 'notebooks':
    os.chdir('../..')

from experiments.notebooks import viz
from experiments.util import get_comparison_result
from experiments.data_util import get_clean_dataset

np.random.seed(0)

In [ ]:
# # from experiments.config.ensemble_config import get_weak_learner_inst_list

# wl_list = get_weak_learner_inst_list([35])[0]
# del wl_list[3]
# wl_list

# # np.random.seed(0)
x, y, feat_names = get_clean_dataset('experiments/data/compas-analysis/compas_two_year_clean.csv')
xtrain, xtest, ytrain, ytest = sk.model_selection.train_test_split(x, y, test_size=0.2, random_state=0)

# m1 = StableLinearClassifier(
#     weak_learners=wl_list,
#     max_complexity = 6,
#     alpha = 1,
#     min_mult = 1,
#     penalty = 'l1',
#     max_rules = None,
#     include_linear = False)
# # m2 = imodels.FPLassoClassifier(alpha=630.957344480193, disc_strategy='simple', max_rules=None, maxcardinality=1, include_linear = False)
# m2 = imodels.RuleFitClassifier(alpha=30, include_linear=False, max_rules=None, n_estimators=3, random_state=0)
# m4 = imodels.BoostedRulesClassifier(n_estimators=5)
# m5 = imodels.SkopeRulesClassifier(max_depth=1, n_estimators=100, precision_min=0.3)

# ms = [m1, m2, m4, m5]
# for model in ms:
#     model.fit(xtrain, ytrain, feature_names=feat_names)

# [m.complexity_ for m in ms]
# [get_best_accuracy(m) for m in ms]

In [ ]:
# m1 = imodels.RuleFitClassifier(alpha=30, n_estimators=2, max_rules=None, random_state=0)
# m2 = imodels.SkopeRulesClassifier(n_estimators=1, random_state=0)
# m1.fit(xtrain, ytrain)
# m2.fit(xtrain, ytrain)

In [ ]:
# accuracy_score(ytest, m1.predict(xtest))

In [ ]:
# m2.rules_

In [ ]:
# from experiments.models.util import extract_ensemble, split

In [ ]:
# extract_ensemble([m1, m2], xtrain, ytrain, min_multiplicity=1)

In [ ]:
# for i, model in enumerate(ms):
#     plt.subplot(2, 2, i + 1)
#     yscores = model.predict_proba(xtest)[:, 1]
#     plt.title(repr(model)[:10] + ' - AP ' + str(average_precision_score(ytest, yscores))[:5])
# #     plt.title(repr(model)[:10] + ' - AUC ' + str(roc_auc_score(ytest, yscores))[:5])
#     y, x, _ = precision_recall_curve(ytest, yscores)
# #     x, y, _ = roc_curve(ytest, model.predict_proba(xtest)[:, 1])
#     plt.step(x, y)
# plt.tight_layout()

In [ ]:
# rule_df = get_comparison_result(MODEL_COMPARISON_PATH, 'stbl_l1_mm1', 'test')['rule_df']
# comp_result = get_comparison_result(MODEL_COMPARISON_PATH, 'stbl_l1_mm1', 'test')
# from experiments.compare_models import compute_meta_auc

In [ ]:
from experiments.util import remove_x_axis_duplicates

In [ ]:
test_mul_curves = get_comparison_result(MODEL_COMPARISON_PATH, 'skope_rules', 'recidivism', 'test')['df']

In [ ]:
curves = test_mul_curves.index.unique()

In [ ]:
curr_x = test_mul_curves[test_mul_curves.index == curves[0]]['mean_complexity']
curr_y = test_mul_curves[test_mul_curves.index == curves[0]][f'mean_rocauc']
curr_x = curr_x[curr_x.argsort()]
curr_y = curr_y[curr_x.argsort()]
remove_x_axis_duplicates(curr_x, curr_y)

In [ ]:
curr_x = test_mul_curves[test_mul_curves.index == curves[1]]['mean_complexity']
curr_y = test_mul_curves[test_mul_curves.index == curves[1]][f'mean_rocauc']
curr_x = curr_x[curr_x.argsort()]
curr_y = curr_y[curr_x.argsort()]
remove_x_axis_duplicates(curr_x, curr_y)

In [ ]:
def merge_overlapping_curves(test_mul_curves):
    final_x = []
    final_y = []
    curves = test_mul_curves.index.unique()

    start_compl = 0
    for i in range(curves.shape[0]):
        curr_x = test_mul_curves[test_mul_curves.index == curves[i]]['mean_complexity']
        curr_y = test_mul_curves[test_mul_curves.index == curves[i]][f'mean_rocauc']
        curr_x, curr_y = curr_x[curr_x.argsort()], curr_y[curr_x.argsort()]
        curr_x, curr_y = remove_x_axis_duplicates(curr_x, curr_y)
        curr_x, curr_y = curr_x[curr_x >= start_compl], curr_y[curr_x >= start_compl]

        if i != curves.shape[0] - 1:
            next_x = test_mul_curves[test_mul_curves.index == curves[i + 1]]['mean_complexity']
            next_y = test_mul_curves[test_mul_curves.index == curves[i + 1]][f'mean_rocauc']
            next_x, next_y = next_x[next_x.argsort()], next_y[next_x.argsort()]
            next_x, next_y = remove_x_axis_duplicates(next_x, next_y)

        found_switch_point = False
        for j in range(curr_x.shape[0] - 1):

            final_x.append(curr_x[j])
            final_y.append(curr_y[j])

            if i != curves.shape[0] - 1:

                next_x_next_val = next_x[next_x > curr_x[j]][0]
                next_y_next_val = next_y[next_x > curr_x[j]][0]
                curr_x_next_val = curr_x[j + 1]
                curr_y_next_val = curr_y[j + 1]

                if next_y_next_val > curr_y_next_val and next_x_next_val - curr_x_next_val <= 5:
                    start_compl = next_x_next_val
                    found_switch_point = True
                    break

        if not found_switch_point:
            return np.array(final_x), np.array(final_y)

    return np.array(final_x), np.array(final_y)

In [ ]:
merge_overlapping_curves(test_mul_curves)

In [ ]:
for i in range(len(curves)):


In [ ]:
# x = np.array([1, 1, 1, 2, 3, 3, 3])
# y = np.array([0, 5, 0, 2, 6, 0, 0])

# y_for_unique_x = []

# unique_arr, inds, counts = np.unique(x, return_index=True, return_counts=True)
# for i, ind in enumerate(inds):
#     y_for_unique_x.append(y[ind:ind+counts[i]].max())
# y_for_unique_x

In [ ]:
# for model in ['stbl_l1_mm1']:
#     for dataset in ['recidivism']:
#         result = pkl.load(open(f'experiments/comparison_data/reg_data/{dataset}/test/{model}_test_comparisons.pkl', 'rb'))
# #         result['estimators'] = list(map(lambda x: x[:-2], result['estimators']))
#         result['rule_df'].index = result['rule_df'].index.str.slice(0, -2)
#         print(result['rule_df'].index)
# #          result['df']['mean_complexity'] = result['df'][f'{dataset}_complexity']
# #         result['meta_auc_df'] = compute_meta_auc(result['df'])
#         pkl.dump(result, open(f'experiments/comparison_data/reg_data/{dataset}/test/{model}_test_comparisons.pkl', 'wb'))

In [ ]:
# pkl.load(open(f'experiments/comparison_data/reg_data/readmission/test/stbl_l2_mm1_test_comparisons.pkl', 'rb'))['df']

In [ ]:
# from slurmpy import Slurm
# partition = 'high'
# s = Slurm("compare_models", {"partition": partition})

# for dataset in ['readmission', 'credit', 'juvenile']:
#     for i in range(36):
#         s.run(f'python experiments/compare_models.py --model stbl_l1 --dataset {dataset} --ignore_cache --parallel_id {i*6} {(i+1)*6-1}')

In [ ]:
from slurmpy import Slurm

partition = 'high'
s = Slurm("compare_models", {"partition": partition})

In [ ]:
# import subprocess

In [ ]:
# for i in range(36):
#     subprocess.run(['scancel', f'{i+956883}'])

In [ ]:
# for model in ['random_forest', 'gradient_boosting', 'skope_rules', 'rulefit', 'fplasso', 'brs']:
#     s.run(f'python experiments/compare_models.py --model {model} --test --dataset juvenile --ignore_cache --low_data')

In [ ]:
# !python experiments/compare_models.py --model gradient_boosting --test --dataset credit --ignore_cache --low_data

In [ ]:
# for i in range(10):
#     s.run(f'python experiments/compare_models.py --model brl --dataset credit --ignore_cache --test --low_data --parallel_id {i}')

In [ ]:
# for model in ['stbl_l1_mm1', 'stbl_l1_mm0', 'stbl_l2_mm1', 'stbl_l2_mm0']:
#     s.run(f'python experiments/compare_models.py --model {model} --test --ensemble --dataset juvenile --ignore_cache --low_data')

In [ ]:
# for i in range(18):
#     s.run(f'python experiments/compare_models.py --model stbl_l1_mm0_e --test --ensemble --dataset readmission --ignore_cache --parallel_id {i}')

# validation plots - readmission

In [ ]:
val_models = ['stbl_l2', 'stbl_l1', 'random_forest', 'gradient_boosting', 'skope_rules', 'rulefit', 'brs', 'fplasso',
              'brl']  #'stbl_l2', 'stbl_l1', 'stbl_unlim_l1', 'stbl_unlim_fps_l1']
val_results = [get_comparison_result(MODEL_COMPARISON_PATH, mname, 'readmission') for mname in val_models]
for result in val_results:
    viz.viz_comparison_val_average(result, metric='mean_rocauc')
    plt.show()

# validation plots - credit default

In [ ]:
val_models = ['stbl_l2', 'stbl_l1', 'random_forest', 'gradient_boosting', 'skope_rules', 'rulefit', 'brs', 'fplasso',
              'brl']  #'stbl_l2', 'stbl_l1', 'stbl_unlim_l1', 'stbl_unlim_fps_l1', 'fplasso', 'brl']
val_results = [get_comparison_result(MODEL_COMPARISON_PATH, mname, 'credit') for mname in val_models]
for result in val_results:
    viz.viz_comparison_val_average(result, metric='mean_avg_precision')
    plt.show()

# validation plots - recidivism

In [ ]:
!python experiments / compare_models.py --model brs --dataset recidivism --ignore_cache

In [ ]:
s.run(f'python experiments/combine.py --test --model brl --dataset recidivism')

In [ ]:
19 * 10

In [ ]:
s.run('python experiments/compare_models.py --test --ensemble --dataset recidivism --ignore_cache')

In [ ]:
for model in ['random_forest', 'gradient_boosting', 'skope_rules', 'rulefit', 'brs', 'fplasso']:
    s.run(f'python experiments/compare_models.py --model {model} --test --dataset recidivism --ignore_cache')

In [ ]:
s.run(f'python experiments/compare_models.py --model brl --test --dataset recidivism --ignore_cache --parallel_id 0 19')
s.run(
    f'python experiments/compare_models.py --model brl --test --dataset recidivism --ignore_cache --parallel_id 20 39')
s.run(
    f'python experiments/compare_models.py --model brl --test --dataset recidivism --ignore_cache --parallel_id 40 59')

In [ ]:
for i in range(20):
    s.run(
        f'''python experiments/compare_models.py --model stbl_l1 --ensemble --dataset recidivism --ignore_cache --parallel_id {i * 19} {(i + 1) * 19 - 1}''')

In [ ]:
# common_kwargs = {'c': 10, 'dataset': 'recidivism'}
# get_best_model_under_complexity(**common_kwargs,
#                                             model_name='skope_rules',
#                                             model_cls=imodels.SkopeRulesClassifier,
#                                             curve_params=[1000],
#                                             kwargs={'max_depth': 3})

In [ ]:
val_models = ['stbl_l2', 'stbl_l1', 'random_forest', 'gradient_boosting', 'skope_rules', 'rulefit', 'brs', 'fplasso',
              'brl']  #'stbl_l2', 'stbl_l1']
val_results = [get_comparison_result(MODEL_COMPARISON_PATH, mname, 'recidivism') for mname in val_models]
for result in val_results:
    viz.viz_comparison_val_average(result, metric='mean_rocauc')
    plt.show()

# validation plots - juvenile

In [ ]:
val_models = ['stbl_l2', 'stbl_l1', 'random_forest', 'gradient_boosting', 'skope_rules', 'rulefit', 'brs', 'fplasso',
              'brl']  #'stbl_l2', 'stbl_l1', 'stbl_unlim_l1', 'stbl_unlim_fps_l1']
val_results = [get_comparison_result(MODEL_COMPARISON_PATH, mname, 'juvenile') for mname in val_models]
for result in val_results:
    viz.viz_comparison_val_average(result, metric='mean_avg_precision')
    plt.show()